IMPLEMENTATION OF THE RLHF PAPER

In [ ]:
!pip install palm-rlhf-pytorch

In [ ]:
import torch
from palm_rlhf_pytorch import PaLM

palm = PaLM(
    num_tokens = 20000,
    dim = 512,
    depth = 12,
    flash_attn = True # https://arxiv.org/abs/2205.14135
).cuda()

seq = torch.randint(0, 20000, (1, 2048)).cuda()

loss = palm(seq, return_loss = True)
loss.backward()

# after much training, you can now generate sequences

generated = palm.generate(2048) # (1, 2048)

/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


AttributeError: 'PaLM' object has no attribute 'save'

In [ ]:
print(generated)

tensor([[17686,  2722,  1536,  ...,  3712, 11977, 12101]], device='cuda:0')


In [ ]:
import torch
from palm_rlhf_pytorch import PaLM, RewardModel

reward_model = RewardModel(
    palm,
    num_binned_output = 5 # say rating from 1 to 5
).cuda()

# mock data

seq = torch.randint(0, 20000, (1, 1024)).cuda()
prompt_mask = torch.zeros(1, 1024).bool().cuda() # which part of the sequence is prompt, which part is response
labels = torch.randint(0, 5, (1,)).cuda()

# train

loss = reward_model(seq, prompt_mask = prompt_mask, labels = labels)
loss.backward()

# after much training

reward = reward_model(seq, prompt_mask = prompt_mask)

In [ ]:
print(reward)

tensor([3], device='cuda:0')


In [ ]:
import torch
from palm_rlhf_pytorch import PaLM, RewardModel, RLHFTrainer

# load your pretrained palm

palm = PaLM(
    num_tokens = 20000,
    dim = 512,
    depth = 12
).cuda()

# palm.load('./path/to/pretrained/palm.pt')

# load your pretrained reward model

reward_model = RewardModel(
    palm,
    num_binned_output = 5
).cuda()

# reward_model.load('./path/to/pretrained/reward_model.pt')

# ready your list of prompts for reinforcement learning

prompts = torch.randint(0, 256, (50000, 512)).cuda() # 50k prompts

# pass it all to the trainer and train

trainer = RLHFTrainer(
    palm = palm,
    reward_model = reward_model,
    prompt_token_ids = prompts
)

trainer.train(num_episodes = 50000)

# then, if it succeeded...
# generate say 10 samples and use the reward model to return the best one

answer = trainer.generate(2048, prompt = prompts[0], num_samples = 10) # (<= 2048,)

episodes:   0%|          | 0/50000 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
100%|██████████| 1540/1540 [02:24<00:00, 10.69it/s]

100%|██████████| 1540/1540 [02:24<00:00, 10.69it/s]

100%|██████████| 1536/1536 [02:24<00:00, 10.66it/s]

100%|██████████| 1536/1536 [02:23<00:00, 10.67it/s]

 89%|████████▉ | 1371/1537 [01:56<00:25,  6.56it/s]